## Import R libraries


In [ ]:
library(lme4)
library(mice)
library(arrow)
library(gt)
library(dplyr)
library(gtsummary)
library(arm)
library(ggplot2)
library(broom.mixed)
library(data.table)
library(lubridate)
library(future.apply)
library(patchwork)
library(rlang)
library(forcats)
library(performance)
library(mitml)

#Nick's packages
#packages <- c("tidyverse","ggthemes","styler","readxl","writexl","DBI","dbplyr","knitr","pandoc","janitor", "data.table", "duckdb","powerjoin","collapse","tidyfast","datapasta","fst","dtplyr","bit64","zoo","fuzzyjoin","arrow","hrbrthemes","here","table1", "rvest", "tidymodels", "pscl", "performance", "report")
packages <- c("future", "progressr", "ggthemes","merTools", "styler", "readxl", "writexl", "openxlsx", "labelled",  "webshot2", "flextable", "tableone", "readr", "DBI", "dbplyr", "pandoc", "janitor", "duckdb", "powerjoin", "collapse", "tidyfast", "datapasta", "fst", "dtplyr", "bit64", "zoo", "fuzzyjoin", "tigris", "hablar", "Hmisc", "table1", "arrow", "RColorBrewer", "viridis", "ggalluvial", "ggfittext", "tinytex", "ggrepel", "survival", "remotes", "ggforce", "tidyr", "tidytext", "Matrix", "lme4", "forestplot", "finalfit", "data.table", "broom.mixed", "car", "jtools", "DataExplorer", "psych", "sjPlot", "GGally", "mice", "haven", "dplyr", "gt", "tidyverse", "gtsummary", "performance", "report", "cluster" )

install_if_missing <- function(package) {
  if (!require(package, character.only = TRUE)) {
    install.packages(package, dependencies = TRUE)
    library(package, character.only = TRUE)
  }
}

sapply(packages, install_if_missing)
options(repr.matrix.max.cols = 1e3)    # Show more columns if needed


## Output Directory

In [ ]:
figures_dir <- "Figures"
dir.create(figures_dir, showWarnings = FALSE)

# Functions


In [ ]:
install_if_missing <- function(package) {
  if (!require(package, character.only = TRUE)) {
    install.packages(package, dependencies = TRUE)
    library(package, character.only = TRUE)
  }
}

prepare_data <- function(df, scale_vars, factor_vars, reference_levels = list(), means = original_means, sds = original_sds) {
  missing_scale  <- setdiff(scale_vars, names(df))
  missing_factor <- setdiff(factor_vars, names(df))
  if (length(missing_scale) > 0) {
    stop(sprintf('Missing scale vars in df: %s', paste(missing_scale, collapse = ', ')))
  }
  if (length(missing_factor) > 0) {
    stop(sprintf('Missing factor vars in df: %s', paste(missing_factor, collapse = ', ')))
  }
  for (var in scale_vars) {
    if (var %in% names(means) && var %in% names(sds) && var %in% names(df)) {
      df[[var]] <- as.numeric(scale(df[[var]], center = means[[var]], scale = sds[[var]]))
    }
  }
  for (var in factor_vars) {
    df[[var]] <- factor(df[[var]])
    if (var %in% names(reference_levels)) {
      df[[var]] <- relevel(df[[var]], ref = reference_levels[[var]])
    }
  }
  return(df)
}

random_effect_aor <- function(
  data,
  re_group_name,
  aor_col_name,
  aor_ci_low_name,
  aor_ci_up_name,
  plot_title,
  x_axis_name,
  figure_name,
  y_min = NULL,
  y_max = NULL) {
  re_group <- rlang::sym(re_group_name)
  aor_col <- rlang::sym(aor_col_name)
  ci_lower_col <- rlang::sym(aor_ci_low_name)
  ci_upper_col <- rlang::sym(aor_ci_up_name)
  data <- data[order(data[[aor_col_name]]), ]
  data[[re_group_name]] <- factor(data[[re_group_name]], levels = data[[re_group_name]])
  data$color <- ifelse(data[[aor_col_name]] >= 1, '#2ca02c', '#dc3912')
  p <- ggplot2::ggplot(data, ggplot2::aes(x = !!re_group, y = !!aor_col)) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = !!ci_lower_col, ymax = !!ci_upper_col), width = 0.2, color = 'gray') +
    ggplot2::geom_point(shape = 21, fill = data$color, color = 'black') +
    ggplot2::geom_hline(yintercept = 1, linetype = "dashed", color = "blue") +
    ggplot2::labs(title = plot_title, x = x_axis_name, y = 'Adjusted Odds Ratio') +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.x = ggplot2::element_blank(),
                   axis.ticks.x = ggplot2::element_blank(),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::guides(color = "none")
  if (!is.null(y_min) || !is.null(y_max)) {
    p <- p + ggplot2::coord_cartesian(ylim = c(y_min, y_max))
  }
  ggplot2::ggsave(figure_name, plot = p, dpi = 600, width = 8, height = 6)
  return(p)
}

ltvv_obs_or_set_proportion_plot <- function(data) {
  df <- data
  bins <- c(-Inf, 8, 9, 10, 11, 12, Inf)
  labels <- c('≤8 cc/kg', '8-9 cc/kg', '9-10 cc/kg', '10-11 cc/kg', '11-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_or_obs_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = 'drop') %>%
    dplyr::group_by(prov_npi_shifted) %>%
    dplyr::mutate(total = sum(count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`, proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  sort_column <- labels[1]
  plot_data <- plot_data %>% dplyr::arrange(.data[[sort_column]])
  long_plot_data <- plot_data %>%
    tidyr::pivot_longer(-prov_npi_shifted, names_to = "TidalVolumeBin", values_to = "Proportion") %>%
    dplyr::mutate(prov_npi_shifted = factor(prov_npi_shifted, levels = unique(plot_data$prov_npi_shifted)),
                  TidalVolumeBin = factor(TidalVolumeBin, levels = rev(labels)))
  colors <- c("#dc3912", "#7b3294", "#377eb8", '#f39c12', "#f4d03f", "#2ca02c")
  p <- ggplot2::ggplot(long_plot_data, ggplot2::aes(y = prov_npi_shifted, x = Proportion, fill = TidalVolumeBin)) +
    ggplot2::geom_bar(stat = "identity", position = "fill", color = NA) +
    ggplot2::scale_fill_manual(values = colors, name = "Percent of Tidal Volume / IBW (cc/kg)") +
    ggplot2::guides(fill = ggplot2::guide_legend(reverse = TRUE)) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.y = ggplot2::element_blank(),
                   axis.ticks.y = ggplot2::element_blank(),
                   panel.grid.major.y = ggplot2::element_blank(),
                   panel.grid.minor.y = ggplot2::element_blank(),
                   legend.position = "top",
                   legend.title = ggplot2::element_text(size = 12),
                   legend.text = ggplot2::element_text(size = 10, hjust = 0),
                   legend.key.size = grid::unit(0.8, "lines"),
                   legend.spacing.y = grid::unit(0.5, "lines"),
                   legend.box.spacing = grid::unit(0.5, "lines"),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::labs(y = 'Provider', x = 'Proportion of Set or Obs Tidal Volume (%)', title = 'Proportion of Set or Obs Tidal Volume by Provider')
  return(p)
}

ltvv8_proportion_plot <- function(data, plot_title = 'Proportion of Set Tidal Volume by Provider') {
  df <- data
  bins <- c(-Inf, 8, 9, 10, 11, 12, Inf)
  labels <- c('\u22648 cc/kg', '8-9 cc/kg', '9-10 cc/kg', '10-11 cc/kg', '11-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = 'drop') %>%
    dplyr::group_by(prov_npi_shifted) %>%
    dplyr::mutate(total = sum(count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`, proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  sort_column <- labels[1]
  plot_data <- plot_data %>% dplyr::arrange(.data[[sort_column]])
  long_plot_data <- plot_data %>%
    tidyr::pivot_longer(-prov_npi_shifted, names_to = "TidalVolumeBin", values_to = "Proportion") %>%
    dplyr::mutate(prov_npi_shifted = factor(prov_npi_shifted, levels = unique(plot_data$prov_npi_shifted)),
                  TidalVolumeBin = factor(TidalVolumeBin, levels = rev(labels)))
  colors <- c("#dc3912", "#7b3294", "#377eb8", '#f39c12', "#f4d03f", "#2ca02c")
  p <- ggplot2::ggplot(long_plot_data, ggplot2::aes(y = prov_npi_shifted, x = Proportion, fill = TidalVolumeBin)) +
    ggplot2::geom_bar(stat = "identity", position = "fill", color = NA) +
    ggplot2::scale_fill_manual(values = colors, name = "Percent of Tidal Volume / IBW (cc/kg)") +
    ggplot2::guides(fill = ggplot2::guide_legend(reverse = TRUE)) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.y = ggplot2::element_blank(),
                   axis.ticks.y = ggplot2::element_blank(),
                   panel.grid.major.y = ggplot2::element_blank(),
                   panel.grid.minor.y = ggplot2::element_blank(),
                   legend.position = "top",
                   legend.title = ggplot2::element_text(size = 12),
                   legend.text = ggplot2::element_text(size = 10, hjust = 0),
                   legend.key.size = grid::unit(0.8, "lines"),
                   legend.spacing.y = grid::unit(0.5, "lines"),
                   legend.box.spacing = grid::unit(0.5, "lines"),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::labs(y = 'Provider', x = 'Proportion of Set Tidal Volume (%)', title = plot_title)
  return(p)
}

# Variant: bins focused on ≤6 cc/kg
ltvv6_proportion_plot <- function(data, plot_title = 'Proportion of Set Tidal Volume by Provider') {
  df <- data
  bins <- c(-Inf, 6, 7, 8, 10, 12, Inf)
  labels <- c('\u22646 cc/kg', '6-7 cc/kg', '7-8 cc/kg', '8-10 cc/kg', '10-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = 'drop') %>%
    dplyr::group_by(prov_npi_shifted) %>%
    dplyr::mutate(total = sum(count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(prov_npi_shifted, `Percent of Tidal Volume / IBW (cc/kg)`, proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  sort_column <- labels[1]
  plot_data <- plot_data %>% dplyr::arrange(.data[[sort_column]])
  long_plot_data <- plot_data %>%
    tidyr::pivot_longer(-prov_npi_shifted, names_to = "TidalVolumeBin", values_to = "Proportion") %>%
    dplyr::mutate(prov_npi_shifted = factor(prov_npi_shifted, levels = unique(plot_data$prov_npi_shifted)),
                  TidalVolumeBin = factor(TidalVolumeBin, levels = rev(labels)))
  colors <- c("#dc3912", "#7b3294", "#377eb8", '#f39c12', "#f4d03f", "#2ca02c")
  p <- ggplot2::ggplot(long_plot_data, ggplot2::aes(y = prov_npi_shifted, x = Proportion, fill = TidalVolumeBin)) +
    ggplot2::geom_bar(stat = "identity", position = "fill", color = NA) +
    ggplot2::scale_fill_manual(values = colors, name = "Percent of Tidal Volume / IBW (cc/kg)") +
    ggplot2::guides(fill = ggplot2::guide_legend(reverse = TRUE)) +
    ggplot2::scale_x_continuous(labels = scales::percent_format(accuracy = 1)) +
    ggplot2::theme_minimal(base_size = 14) +
    ggplot2::theme(axis.text.y = ggplot2::element_blank(),
                   axis.ticks.y = ggplot2::element_blank(),
                   panel.grid.major.y = ggplot2::element_blank(),
                   panel.grid.minor.y = ggplot2::element_blank(),
                   legend.position = "top",
                   legend.title = ggplot2::element_text(size = 12),
                   legend.text = ggplot2::element_text(size = 10, hjust = 0),
                   legend.key.size = grid::unit(0.8, "lines"),
                   legend.spacing.y = grid::unit(0.5, "lines"),
                   legend.box.spacing = grid::unit(0.5, "lines"),
                   plot.title = ggplot2::element_text(hjust = 0.5)) +
    ggplot2::labs(y = 'Provider', x = 'Proportion of Set Tidal Volume (%)', title = plot_title)
  return(p)
}

pool_fixed_effects <- function(model_obj,
                               sds        = if (exists("original_sds") && exists("use_scaling") && use_scaling) original_sds else NULL,
                               vars_to_bt = if (exists("scale_vars"))   scale_vars   else NULL) {
  # Back-transforms scaled coefficients to clinical units only when use_scaling == TRUE.
  # β_original = β_scaled / SD  →  OR_original = exp(β_scaled / SD)
  # CI uses Barnard-Rubin df from testEstimates for consistency with pooled p-values.
  mitml_res <- mitml::testEstimates(model = model_obj, extra.pars = FALSE)
  fe  <- mitml_res$estimates
  est <- fe[, "Estimate"]
  se  <- fe[, "Std.Error"]
  df  <- fe[, "df"]

  if (!is.null(sds) && !is.null(vars_to_bt)) {
    for (var in intersect(vars_to_bt, names(sds))) {
      if (var %in% names(est)) {
        est[var] <- est[var] / sds[[var]]
        se[var]  <- se[var]  / sds[[var]]
      }
    }
  }

  t_crit <- stats::qt(0.975, df)

  data.frame(
    Term     = rownames(fe),
    OR       = exp(est),
    Lower_CI = exp(est - t_crit * se),
    Upper_CI = exp(est + t_crit * se),
    P_value  = fe[, "P(>|t|)"],
    row.names = NULL
  )
}

create_fe_table <- function(fe_df,
                            title,
                            output_file = NULL,
                            drop_intercept = TRUE,
                            term_map = NULL,
                            digits_or = 2,
                            digits_ci = 2) {
  stopifnot(is.data.frame(fe_df),
            all(c("Term","OR","Lower_CI","Upper_CI","P_value") %in% names(fe_df)))
  df <- fe_df
  if (drop_intercept) {
    df <- df %>% dplyr::filter(!grepl("^\\(Intercept\\)$", Term))
  }
  # Drop variance-component rows from extra.pars (safety net)
  df <- df %>% dplyr::filter(!grepl("^(Var|Cov)[0-9]+\\[", Term))
  if (is.null(term_map) && exists('rename_terms')) {
    term_map <- rename_terms
  }
  if (!is.null(term_map)) {
    df <- df %>% dplyr::mutate(Term = ifelse(Term %in% names(term_map), term_map[Term], Term))
  }
  if (exists('custom_order')) {
    df <- df %>% dplyr::mutate(Term = factor(Term, levels = custom_order)) %>% dplyr::arrange(Term) %>% dplyr::mutate(Term = as.character(Term))
  }
  df_fmt <- df %>%
    dplyr::mutate(
      `OR (95% CI)` = sprintf(paste0("%.", digits_or, "f (%.", digits_ci, "f, %.", digits_ci, "f)"), OR, Lower_CI, Upper_CI),
      `p-value` = gtsummary::style_pvalue(P_value)
    ) %>%
    dplyr::select(Term, `OR (95% CI)`, `p-value`)
  tab <- df_fmt %>%
    gt::gt(rowname_col = NULL) %>%
    gt::tab_header(title = gt::md(title)) %>%
    gt::fmt_markdown(columns = dplyr::everything()) %>%
    gt::tab_options(table.font.size = gt::px(14)) %>%
    gt::opt_row_striping() %>%
    gt::opt_vertical_padding(scale = 0.5)
  if (!is.null(output_file)) {
    html_code <- gt::as_raw_html(tab)
    writeLines(html_code, output_file)
    message("Table saved to: ", output_file)
  }
  tab
}

create_forest_plot <- function(data,
                               main_breaks = c(0, 2, 4, 6),
                               main_limits = c(0, 6),
                               year_breaks = c(0, 5, 10, 15, 20),
                               year_limits = c(0, 20),
                               filename = "fixed_effects.tiff",
                               width = 8,
                               height = 4,
                               dpi = 600) {
  plot_data <- data |>
    dplyr::filter(Term != "(Intercept)") |>
    dplyr::rename(term = Term, OR = OR, CI_lower = `Lower_CI`, CI_upper = `Upper_CI`) |>
    dplyr::mutate(term_chr = as.character(term),
                  mapped = if (exists("rename_terms")) unname(rename_terms[term_chr]) else NA_character_,
                  term_pretty = dplyr::coalesce(mapped, term_chr),
                  plot_group = ifelse(grepl("^Year:", term_pretty), "Year Effects", "Main Predictors")) |>
    dplyr::select(-mapped) |>
    dplyr::mutate(term_pretty = if (exists("custom_order")) factor(term_pretty, levels = rev(intersect(custom_order, unique(term_pretty)))) else factor(term_pretty, levels = rev(unique(term_pretty))))
  main_effects <- plot_data %>% dplyr::filter(plot_group == "Main Predictors")
  year_effects <- plot_data %>% dplyr::filter(plot_group == "Year Effects")
  plot_main <- ggplot2::ggplot(main_effects, ggplot2::aes(x = term_pretty, y = OR)) +
    ggplot2::geom_point(size = 2) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = CI_lower, ymax = CI_upper), width = 0.2) +
    ggplot2::geom_hline(yintercept = 1, linetype = "dashed", color = "black") +
    ggplot2::coord_flip(clip = "off") +
    ggplot2::scale_y_continuous(breaks = main_breaks, limits = main_limits, expand = ggplot2::expansion(mult = c(0, 0))) +
    ggplot2::labs(title = "Patient and Hospital", x = "Fixed Effect", y = "Odds Ratio") +
    ggplot2::theme_minimal()
  plot_years <- ggplot2::ggplot(year_effects, ggplot2::aes(x = term_pretty, y = OR)) +
    ggplot2::geom_point(size = 2) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = CI_lower, ymax = CI_upper), width = 0.2) +
    ggplot2::geom_hline(yintercept = 1, linetype = "dashed", color = "black") +
    ggplot2::coord_flip(clip = "off") +
    ggplot2::scale_y_continuous(breaks = year_breaks, limits = year_limits, expand = ggplot2::expansion(mult = c(0, 0))) +
    ggplot2::labs(title = "Year", x = "Fixed Effect", y = "Odds Ratio") +
    ggplot2::theme_minimal()
  combined_plot <- plot_main + plot_years + patchwork::plot_layout(nrow = 1) + patchwork::plot_annotation(tag_levels = 'A')
  ggplot2::ggsave(filename, plot = combined_plot, dpi = dpi, width = width, height = height)
  message("Plot saved to: ", filename)
  return(combined_plot)
}

extract_random_effects <- function(model, re_group = "prov_npi_shifted") {
  ranefs <- lme4::ranef(model)[[re_group]]
  se_physician <- attr(lme4::ranef(model, condVar = TRUE)[[re_group]], "postVar")[1, 1, ]
  aor <- exp(ranefs[, "(Intercept)"])
  ci_lower <- exp(ranefs[, "(Intercept)"] - 1.96 * sqrt(se_physician))
  ci_upper <- exp(ranefs[, "(Intercept)"] + 1.96 * sqrt(se_physician))
  data.frame(prov_npi_shifted = rownames(ranefs), AOR = aor, CI_lower = ci_lower, CI_upper = ci_upper)
}

ltvv_median_iqr <- function(data, adherence_bins = c("<8 cc/kg")) {
  df <- data
  bins <- c(-Inf, 8, 9, 10, 11, 12, Inf)
  labels <- c('≤8 cc/kg', '8-9 cc/kg', '9-10 cc/kg', '10-11 cc/kg', '11-12 cc/kg', '>12 cc/kg')
  df <- df %>% dplyr::mutate(`Percent of Tidal Volume / IBW (cc/kg)` = cut(`tidal_volume_set_ibw`, breaks = bins, labels = labels, right = TRUE, include.lowest = TRUE))
  grouped <- df %>%
    dplyr::group_by(.data$prov_npi_shifted, .data$`Percent of Tidal Volume / IBW (cc/kg)`) %>%
    dplyr::summarise(count = dplyr::n(), .groups = "drop") %>%
    dplyr::group_by(.data$prov_npi_shifted) %>%
    dplyr::mutate(total = sum(.data$count), proportion = count / total * 100) %>%
    dplyr::ungroup()
  plot_data <- grouped %>%
    dplyr::select(.data$prov_npi_shifted, .data$`Percent of Tidal Volume / IBW (cc/kg)`, .data$proportion) %>%
    tidyr::pivot_wider(names_from = `Percent of Tidal Volume / IBW (cc/kg)`, values_from = proportion, values_fill = 0)
  adherence_bins <- intersect(adherence_bins, colnames(plot_data))
  adherence <- rowSums(dplyr::select(plot_data, dplyr::all_of(adherence_bins)), na.rm = TRUE)
  med <- stats::median(adherence, na.rm = TRUE)
  q <- stats::quantile(adherence, probs = c(0.25, 0.75), na.rm = TRUE, type = 7)
  message(sprintf("Provider LTVV adherence (%% in %s): median %.1f%% [IQR %.1f?%.1f%%] (n=%d providers)", paste(adherence_bins, collapse = " + "), med, q[[1]], q[[2]], nrow(plot_data)))
  return(tibble::tibble(n_providers = nrow(plot_data), adherence_bins = paste(adherence_bins, collapse = " + "), median = med, q1 = q[[1]], q3 = q[[2]]))
}

summarize_model <- function(model_list, data, outcome, model_name, random_effect_var = "prov_npi_shifted") {
  # --- pull model spec from first fitted model ---
  get_model_spec <- function(model) {
    f <- formula(model)
    # outcome (LHS)
    outcome_name <- all.vars(lme4::nobars(f)[[2]])
    # fixed-effect RHS (covariates)
    rhs_fixed <- lme4::nobars(f)[[3]]
    covariates <- setdiff(all.vars(rhs_fixed), outcome_name)
    # random effects terms
    random_effects <- vapply(lme4::findbars(f), function(b) deparse(b[[3]]), character(1))

    list(
      outcome = outcome_name,
      random_effect = random_effects,
      covariates = covariates
    )
  }

  spec <- get_model_spec(model_list[[1]])

  # --- existing summary calculations ---
  log_mor_vec <- c()
  var_log_mor_vec <- c()
  for (model in model_list) {
    vc_df <- as.data.frame(VarCorr(model))
    variance_random_intercepts <- vc_df$vcov[1]
    sd_random_intercepts <- vc_df$sdcor[1]
    n_clusters <- nrow(lme4::ranef(model)[[random_effect_var]])
    sem_sd <- sd_random_intercepts / sqrt(n_clusters)
    var_sem <- 2 * variance_random_intercepts * (sem_sd^2)
    mor <- exp(sqrt(2 * variance_random_intercepts) * 0.6745)
    log_mor <- log(mor)
    deriv_log_mor <- 0.6745 / sqrt(2 * variance_random_intercepts)
    var_log_mor <- (deriv_log_mor^2) * var_sem
    log_mor_vec <- c(log_mor_vec, log_mor)
    var_log_mor_vec <- c(var_log_mor_vec, var_log_mor)
  }
  m <- length(log_mor_vec)
  Q_bar <- mean(log_mor_vec)
  W <- mean(var_log_mor_vec)
  B <- var(log_mor_vec)
  Tval <- W + (1 + 1/m) * B
  df <- (m - 1) * (1 + (W / ((1 + 1/m) * B)))^2
  crit <- stats::qt(0.975, df)
  log_mor_lower <- Q_bar - crit * sqrt(Tval)
  log_mor_upper <- Q_bar + crit * sqrt(Tval)
  mor_pooled <- exp(Q_bar)
  mor_lower <- exp(log_mor_lower)
  mor_upper <- exp(log_mor_upper)

  # Pool ICC on logit scale (keeps CI within [0,1])
  logit_icc_vec <- c()
  var_logit_icc_vec <- c()
  for (model in model_list) {
    var_re <- as.numeric(as.data.frame(VarCorr(model))$vcov[1])
    var_res <- pi^2 / 3
    icc <- var_re / (var_re + var_res)
    n_clusters <- nrow(lme4::ranef(model)[[random_effect_var]])
    se_icc <- sqrt((2 * icc^2 * (1 - icc)^2) / (n_clusters - 1))
    logit_icc <- log(icc / (1 - icc))
    var_logit_icc <- se_icc^2 / (icc * (1 - icc))^2
    logit_icc_vec <- c(logit_icc_vec, logit_icc)
    var_logit_icc_vec <- c(var_logit_icc_vec, var_logit_icc)
  }
  m_i <- length(logit_icc_vec)
  Qb_i <- mean(logit_icc_vec)
  W_i <- mean(var_logit_icc_vec)
  B_i <- var(logit_icc_vec)
  T_i <- W_i + (1 + 1/m_i) * B_i
  df_i <- (m_i - 1) * (1 + (W_i / ((1 + 1/m_i) * B_i)))^2
  crit_i <- stats::qt(0.975, df_i)
  logit_lower <- Qb_i - crit_i * sqrt(T_i)
  logit_upper <- Qb_i + crit_i * sqrt(T_i)
  icc_pooled <- exp(Qb_i) / (1 + exp(Qb_i))
  icc_lower  <- exp(logit_lower) / (1 + exp(logit_lower))
  icc_upper  <- exp(logit_upper) / (1 + exp(logit_upper))

  baseline_prob <- mean(as.numeric(as.character(data[[outcome]])))
  baseline_odds <- baseline_prob / (1 - baseline_prob)
  new_odds <- baseline_odds * mor_pooled
  new_prob <- new_odds / (1 + new_odds)
  absolute_risk_diff <- new_prob - baseline_prob
  NNT <- ifelse(absolute_risk_diff == 0, Inf, abs(1 / absolute_risk_diff))

  n_hospitalizations <- length(unique(data$hospitalizations_joined_id))
  n_patients <- length(unique(data$mdm_link_id))
  n_providers <- length(unique(data[[random_effect_var]]))
  n_observations <- nrow(data)

  summary <- list(
    model_name = model_name,
    # model spec (from formula)
    outcome = spec$outcome,
    random_effect = spec$random_effect,
    covariates = spec$covariates,
    # existing metrics
    icc = icc_pooled,
    icc_ci_lower = icc_lower,
    icc_ci_upper = icc_upper,
    mor = mor_pooled,
    mor_ci_lower = mor_lower,
    mor_ci_upper = mor_upper,
    baseline_prob = baseline_prob,
    new_prob = new_prob,
    absolute_risk_diff = absolute_risk_diff,
    nnt = NNT,
    n_hospitalizations = n_hospitalizations,
    n_patients = n_patients,
    n_providers = n_providers,
    n_observations = n_observations
  )
  return(summary)
}

create_model_summary_html <- function(model_summaries, output_file = "model_summary.html") {
  if (inherits(model_summaries, "model_summary")) {
    model_summaries <- list(model_summaries)
  }
  stopifnot(is.list(model_summaries), length(model_summaries) >= 1)

  fmt_pct <- function(x, digits = 1) sprintf(paste0("%.", digits, "f%%"), 100 * x)
  fmt_num <- function(x, digits = 3) sprintf(paste0("%.", digits, "f"), x)
  fmt_ci  <- function(est, lo, hi, digits = 3) sprintf(paste0("%.", digits, "f (%.", digits, "f - %.", digits, "f)"), est, lo, hi)
  fmt_int <- function(x) format(x, big.mark = ",", scientific = FALSE)
  fmt_nnt <- function(x) ifelse(is.finite(x), sprintf("%.1f", x), "&infin;")

  html_content <- paste0(
'<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Model Summary Report</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 40px; color: #2c3e50; }
    h1 { color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 8px; }
    h2 { color: #34495e; margin-top: 28px; }
    h3 { color: #34495e; margin-top: 18px; }
    table { border-collapse: collapse; width: 100%; margin: 16px 0; }
    th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
    th { background-color: #f4f6f8; font-weight: bold; }
    tr:nth-child(even) { background-color: #fafbfc; }
    .metric { font-weight: bold; color: #2980b9; }
    .summary-box { background-color: #ecf0f1; padding: 14px; margin: 12px 0; border-radius: 6px; }
    .small { color: #6b7280; font-size: 0.9em; }
  </style>
</head>
<body>
  <h1>Model Summary Report</h1>
  <p class="small"><strong>Generated on:</strong> ', format(Sys.time(), "%B %d, %Y %H:%M %Z"), '</p>
  <h2>Model Comparison Summary</h2>
  <table>
    <tr>
      <th>Model</th>
      <th>ICC (95% CI)</th>
      <th>MOR (95% CI)</th>
      <th>Baseline Risk</th>
      <th>Risk Difference</th>
      <th>NNT (illustrative)</th>
      <th>Observations</th>
      <th>Providers</th>
    </tr>'
  )

  for (summary in model_summaries) {
    req <- c("model_name","icc","icc_ci_lower","icc_ci_upper",
             "mor","mor_ci_lower","mor_ci_upper",
             "baseline_prob","absolute_risk_diff","nnt",
             "n_observations","n_providers")
    missing_fields <- setdiff(req, names(summary))
    if (length(missing_fields) > 0) {
      stop(sprintf("Missing fields in a model summary: %s", paste(missing_fields, collapse = ", ")))
    }

    html_content <- paste0(html_content, sprintf('
    <tr>
      <td><strong>%s</strong></td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
      <td>%s</td>
    </tr>',
      summary$model_name,
      fmt_ci(summary$icc, summary$icc_ci_lower, summary$icc_ci_upper, digits = 3),
      fmt_ci(summary$mor, summary$mor_ci_lower, summary$mor_ci_upper, digits = 3),
      fmt_pct(summary$baseline_prob, 1),
      fmt_pct(summary$absolute_risk_diff, 2),
      fmt_nnt(summary$nnt),
      fmt_int(summary$n_observations),
      fmt_int(summary$n_providers)
    ))
  }

  html_content <- paste0(html_content, '
  </table>
')

  for (summary in model_summaries) {
    req2 <- c("model_name","icc","icc_ci_lower","icc_ci_upper",
              "mor","mor_ci_lower","mor_ci_upper",
              "baseline_prob","new_prob","absolute_risk_diff","nnt",
              "n_hospitalizations","n_patients","n_providers","n_observations")
    missing_fields <- setdiff(req2, names(summary))
    if (length(missing_fields) > 0) {
      stop(sprintf("Missing fields in a model summary (detailed section): %s", paste(missing_fields, collapse = ", ")))
    }

    req3 <- c("outcome","random_effect","covariates")
    missing_fields <- setdiff(req3, names(summary))
    if (length(missing_fields) > 0) {
      stop(sprintf("Missing model spec fields: %s", paste(missing_fields, collapse = ", ")))
    }

    html_content <- paste0(html_content, sprintf('
  <h2>%s - Detailed Results</h2>
  <div class="summary-box">
    <h3>Model Specification</h3>
    <p><span class="metric">Outcome:</span> %s</p>
    <p><span class="metric">Random Effect:</span> %s</p>
    <p><span class="metric">Covariates:</span> %s</p>
  </div>
  <div class="summary-box">
    <h3>Intraclass Correlation Coefficient (ICC)</h3>
    <p><span class="metric">Pooled ICC:</span> %s</p>

    <h3>Median Odds Ratio (MOR)</h3>
    <p><span class="metric">Pooled MOR:</span> %s</p>

    <h3>Risk Assessment</h3>
    <p><span class="metric">Baseline Probability:</span> %s (%s)</p>
    <p><span class="metric">New Probability with MOR:</span> %s (%s)</p>
    <p><span class="metric">Absolute Risk Difference:</span> %s</p>
    <p><span class="metric">Number Needed to Treat (NNT):</span> %s</p>

    <h3>Dataset Summary</h3>
    <p><span class="metric">Hospitalizations:</span> %s</p>
    <p><span class="metric">Patients:</span> %s</p>
    <p><span class="metric">Providers:</span> %s</p>
    <p><span class="metric">Total Observations:</span> %s</p>
  </div>',
      summary$model_name,
      summary$outcome,
      paste(summary$random_effect, collapse = ", "),
      paste(summary$covariates, collapse = ", "),
      fmt_ci(summary$icc, summary$icc_ci_lower, summary$icc_ci_upper, digits = 3),
      fmt_ci(summary$mor, summary$mor_ci_lower, summary$mor_ci_upper, digits = 3),
      fmt_num(summary$baseline_prob, 3), fmt_pct(summary$baseline_prob, 1),
      fmt_num(summary$new_prob, 3), fmt_pct(summary$new_prob, 1),
      fmt_pct(summary$absolute_risk_diff, 2),
      fmt_nnt(summary$nnt),
      fmt_int(summary$n_hospitalizations),
      fmt_int(summary$n_patients),
      fmt_int(summary$n_providers),
      fmt_int(summary$n_observations)
    ))
  }

  html_content <- paste0(html_content, '
</body>
</html>')

  writeLines(html_content, output_file)
  message("HTML summary saved to: ", output_file)
  invisible(html_content)
}

fit_glmer_model <- function(data_list, explanatory_vars, outcome, random_effect = "prov_npi_shifted") {
  formula_text <- paste(outcome, "~", paste(explanatory_vars, collapse = " + "),
                        "+ (1 |", random_effect, ")")
  message(paste("Formula:", formula_text))
  future_lapply(data_list, function(data) {
    fit <- glmer(as.formula(formula_text), data = data, family = binomial("logit"),
                 control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e6)))
    # Gradient restart: if bobyqa produced convergence warnings, re-optimise from the
    # converged parameter vector with a 10x larger iteration budget.  This is the
    # canonical lme4 fix for near-separation / Hessian issues (Ben Bolker FAQ).
    if (isTRUE(length(fit@optinfo$conv$lme4$messages) > 0)) {
      ss <- lme4::getME(fit, c("theta", "fixef"))
      fit2 <- tryCatch(
        update(fit, start = ss,
               control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e7))),
        error = function(e) { message("Restart failed: ", e$message); fit }
      )
      if (length(fit2@optinfo$conv$lme4$messages) < length(fit@optinfo$conv$lme4$messages)) {
        fit <- fit2
      }
    }
    fit
  }, future.seed = 42L)
}


<h2> Rename and reorder terms </h2>

In [ ]:
# Create a mapping from current term names to display names
rename_terms <- c(
  "(Intercept)" = "(Intercept)",
  "age" = "Age",
  
  "race_categoryAmerican Indian or Alaska Native" = "Race: American Indian or Alaska Native",
  "race_categoryAsian" = "Race: Asian",
  "race_categoryBlack or African-American" = "Race: Black or African-American",
  "race_categoryBlack or African American" = "Race: Black or African-American",
  "race_categoryNative Hawaiian or Other Pacific Islander" = "Race: Native Hawaiian or Other Pacific Islander",
  "race_categoryOther" = "Race: Other",
  "race_categoryUnknown" = "Race: Unknown",
  
  "sex_categoryMale" = "Sex: Male",
  "height_cm" = "Height (cm)",

  # ICU type (Task 2; reference = mixed)
  "icu_type_5catcardiac"    = "ICU Type: Cardiac",
  "icu_type_5catmedical"    = "ICU Type: Medical",
  "icu_type_5catneurologic" = "ICU Type: Neurologic",
  "icu_type_5catsurgical"   = "ICU Type: Surgical",
  
  # Era entries (Task 5 collapse: 2011-2015 is reference level)
  "era2016-2019" = "Year: 2016\u20132019",
  "era2020-2022" = "Year: 2020\u20132022",
  "era2023-2025" = "Year: 2023\u20132025",
  
  "elix_vw" = "Elixhauser Score",
  
  "hospital_idHospital 1" = "Hospital 1",
  "hospital_idHospital 2" = "Hospital 2",
  "hospital_idHospital 3" = "Hospital 3",
  "hospital_idHospital 4" = "Hospital 4",
  "hospital_idHospital 5" = "Hospital 5",
  "hospital_idHospital 6" = "Hospital 6",
  "hospital_idHospital 7" = "Hospital 7",
  "hospital_idHospital 8" = "Hospital 8",
  "hospital_idHospital 9" = "Hospital 9",
  
  "bmi_calc" = "BMI",
  "sf_ratio" = "SF Ratio",
  "pco2" = "PCO2",
  "ph" = "pH",
  "laps2" = "LAPS2 Score",
  "covidTRUE" = 'SARS-CoV-2'
)

custom_order <- c(
  "(Intercept)",
  "Age",
  "BMI",
  
  "Race: American Indian or Alaska Native",
  "Race: Asian",
  "Race: Black or African-American",
  "Race: Native Hawaiian or Other Pacific Islander",
  "Race: Other",
  "Race: Unknown",
  
  "Sex: Male",
  "Height (cm)",
  
  "Elixhauser Score",
  "LAPS2 Score",

  "SF Ratio",
  "PCO2",
  "pH",

  "SARS-CoV-2",
    
  "Hospital 1",
  "Hospital 2",
  "Hospital 3",
  "Hospital 4",
  "Hospital 5",
  "Hospital 6",
  "Hospital 7",
  "Hospital 8",
  "Hospital 9",

  "ICU Type: Cardiac",
  "ICU Type: Medical",
  "ICU Type: Neurologic",
  "ICU Type: Surgical",

  "Year: 2016\u20132019",
  "Year: 2020\u20132022",
  "Year: 2023\u20132025"
)

<h2> Read in LPV full dataset </h2>


In [ ]:
data <- read_parquet('intermediate_outputs/LPV_final_data.parquet')

# Print counts
print(length(unique(data$hospitalizations_joined_id)))
print(length(unique(data$prov_npi_shifted)))
print(nrow(data))
# head(data,10)

<h3> MICE imputation with Rubin's Rules </h3>

In [ ]:
# Variable definitions for scaling and factor levels
use_scaling <- TRUE  # TRUE = z-score all scale_vars; FALSE = clinical units only
scale_vars  <- c('age', 'bmi_calc', 'height_cm', 'elix_vw', 'laps2', 'ph', 'pco2', 'sf_ratio')

# Note: dataset has ltvv_6/ltvv_8 and ltvv_tidal_volume_set_or_obs_6/_8 (not the generic name)
factor_vars <- c('ltvv_6', 'ltvv_8', 'ltvv_tidal_volume_set_or_obs_6', 'ltvv_tidal_volume_set_or_obs_8',
  'race_category', 'sex_category', 'recorded_year', 'hospital_id', 'icu_type_5cat',
  'prov_npi_shifted', 'mdm_link_id', 'covid')

reference_levels <- list(
  race_category = 'White',
  sex_category  = 'Female',
  hospital_id   = 'Hospital Ref',
  icu_type_5cat = 'mixed'
)

In [ ]:
# Task 7: Rescale age to decades and sf_ratio to per-10 units (meeting 2026-06-12)
# These are NOT z-scored; 1 model unit = 1 decade of age / 10 sf_ratio units
# Must run before data_impute split so MICE imputes on the rescaled scale
data$age      <- data$age / 10
data$sf_ratio <- data$sf_ratio / 10

# 1. Define your imputation variables
vars_for_impute <- c("age", "sex_category", "height_cm", "bmi_calc", "elix_vw", "laps2", "ph", "pco2", "sf_ratio", "hospital_id")
# Split into imputed and non-imputed data
data_impute <- data[, vars_for_impute]
data_nonimpute <- data[, !(names(data) %in% vars_for_impute)]

# Save pre-imputation data — used in Task 9 CC analysis to identify originally-complete rows
data_preimpute <- data

# Get original means and SDs for imputation variables - need them for scaling after imputation
original_means <- sapply(data[scale_vars], function(x) mean(x, na.rm = TRUE))
original_sds   <- sapply(data[scale_vars], function(x) sd(x, na.rm = TRUE))

print(length(rownames(data_impute)))
print(summary(data_impute))
print(sapply(data_impute, function(x) mean(is.na(x)) * 100))

In [ ]:
# If variable needs to be factorized, do so prior to imputation
for (var in factor_vars) {
  if (var %in% names(data_impute)) {
    data_impute[[var]] <- factor(data_impute[[var]])
  }
}

# 3. Run MICE only on imputation variables
imp <- mice(data_impute, m = 5, method = "pmm", seed = 123)

# 4. Combine imputed datasets with non-imputed columns
completed_data_list <- lapply(1:5, function(i) {
  data_combined <- cbind(complete(imp, i), data_nonimpute)
  prepare_data(
    data_combined, 
    if (use_scaling) scale_vars else character(0), 
    factor_vars, 
    reference_levels, 
    means = original_means, 
    sds = original_sds
  )  # Apply scaling + factor transformations
})

head(completed_data_list[[1]], 5)

<h1> Create datasets for each model </h1>

In [ ]:
# Define cohorts from imputed datasets (list of 5)
mv_data_list <- completed_data_list

ahrf_data_list <- lapply(completed_data_list, function(df) {
  df %>% dplyr::filter(cohort_eligible == 1L)
})

# Wrap in mitml.list for downstream pooled modeling
mv_data  <- as.mitml.list(mv_data_list)
ahrf_data <- as.mitml.list(ahrf_data_list)

# Pre-imputation AHRF subset — used in Task 9 CC analysis to identify originally-complete rows
ahrf_data_preimpute <- data_preimpute %>% dplyr::filter(cohort_eligible == 1L)

cat(sprintf('Total ahrf datasets: %d (each aligned to completed_data_list)
', length(ahrf_data)))

In [ ]:
# Derive initial and subsequent datasets for MV and ahrf (lists of 5)
initial_mv_data_list <- lapply(mv_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == TRUE)
})

subsequent_mv_data_list <- lapply(mv_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == FALSE)
})

initial_ahrf_data_list <- lapply(ahrf_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == TRUE)
})

subsequent_ahrf_data_list <- lapply(ahrf_data, function(df) {
  df %>% dplyr::filter(initial_vent_day == FALSE)
})

initial_mv_data       <- as.mitml.list(initial_mv_data_list)
subsequent_mv_data    <- as.mitml.list(subsequent_mv_data_list)
initial_ahrf_data      <- as.mitml.list(initial_ahrf_data_list)
subsequent_ahrf_data   <- as.mitml.list(subsequent_ahrf_data_list)

cat(sprintf('Initial MV datasets: %d; Subsequent MV datasets: %d\n', length(initial_mv_data), length(subsequent_mv_data)))
cat(sprintf('Initial ahrf datasets: %d; Subsequent ahrf datasets: %d\n', length(initial_ahrf_data), length(subsequent_ahrf_data)))
cat(sprintf('Rows in initial MV dataset 1: %d\n', nrow(initial_mv_data[[1]])))
cat(sprintf('Rows in subsequent MV dataset 1: %d\n', nrow(subsequent_mv_data[[1]])))
cat(sprintf('Rows in initial ahrf dataset 1: %d\n', nrow(initial_ahrf_data[[1]])))
cat(sprintf('Rows in subsequent ahrf dataset 1: %d\n', nrow(subsequent_ahrf_data[[1]])))


In [ ]:
# Task 5 — Era collapse: 15 annual year indicators → 4 eras
# Triggered by persistent convergence warnings (degenerate Hessian) confirming complete
# separation in sparse early-year cells.  Eras per CLAUDE.md Task 5:
#   2011-2015 | 2016-2019 | 2020-2022 | 2023-2025
add_era <- function(df) {
  yr <- as.integer(as.character(df$recorded_year))
  df$era <- factor(
    dplyr::case_when(
      yr <= 2015 ~ "2011-2015",
      yr <= 2019 ~ "2016-2019",
      yr <= 2022 ~ "2020-2022",
      TRUE       ~ "2023-2025"
    ),
    levels = c("2011-2015", "2016-2019", "2020-2022", "2023-2025")
  )
  df
}

ahrf_data            <- as.mitml.list(lapply(ahrf_data,            add_era))
mv_data              <- as.mitml.list(lapply(mv_data,              add_era))
initial_ahrf_data    <- as.mitml.list(lapply(initial_ahrf_data,    add_era))
subsequent_ahrf_data <- as.mitml.list(lapply(subsequent_ahrf_data, add_era))
initial_mv_data      <- as.mitml.list(lapply(initial_mv_data,      add_era))
subsequent_mv_data   <- as.mitml.list(lapply(subsequent_mv_data,   add_era))

cat("Era distribution in AHRF cohort (imputation 1):\n")
print(table(ahrf_data[[1]]$era))


In [ ]:
ahrf6_data <- ahrf_data
ahrf8_data <- ahrf_data

<h1> Figures 2 and 3 </h1>

-  ahrf6 - Tidal Volume Set by Provider - two panel
-  ahrf6 - Fixed effects plot

<h2> Fit ahrf6 model </h2>

In [ ]:
# Cap/raise threads per worker; try 2 to lift CPU a bit
Sys.setenv(
  OMP_NUM_THREADS = "2",
  MKL_NUM_THREADS = "2",
  OPENBLAS_NUM_THREADS = "2",
  VECLIB_MAXIMUM_THREADS = "2",
  NUMEXPR_NUM_THREADS = "2"
)

# Use as many workers as helpful, capped by cores and list length
max_workers <- min(length(ahrf_data), parallelly::availableCores())  # if 5 items, this gives up to 5
future::plan(multisession, workers = max_workers)

In [ ]:
# Check all downstream variables for missingness
vars_all <- unique(c(
  "age","race_category","sex_category","height_cm","era","elix_vw",
  "hospital_id","icu_type_5cat","bmi_calc","sf_ratio","pco2","ph","laps2","covid",
  "ltvv_6","ltvv_8","ltvv_tidal_volume_set_or_obs_6","ltvv_tidal_volume_set_or_obs_8",
  "prov_npi_shifted"
))

df <- mv_data[[1]]  # or ahrf_data[[1]]

na_counts <- sapply(df[vars_all], function(x) sum(is.na(x)))
na_pct    <- sapply(df[vars_all], function(x) mean(is.na(x)) * 100)

data.frame(var = vars_all, na = na_counts, na_pct = round(na_pct, 2))


In [ ]:
# Primary AHRF-6 covariate set (shared by overall, day-1, subsequent models and downstream cells)
ahrf6_vars <- c("age", "race_category", "sex_category", "height_cm", "era", "elix_vw",
                "hospital_id", "icu_type_5cat", "bmi_calc", "sf_ratio", "pco2", "ph", "laps2")

message("Fitting ahrf6 overall model...")
ahrf6_model <- fit_glmer_model(ahrf_data, ahrf6_vars, "ltvv_6")
message("ahrf6 overall model fit!")


<h2> Get fixed effects </h2>

In [ ]:
ahrf6_fe <- pool_fixed_effects(ahrf6_model)
ahrf6_fe_table <- create_fe_table(
  fe_df = ahrf6_fe,
  title = "**AHRF-6 Model - Pooled Fixed Effects**",
  output_file = file.path(figures_dir, "ahrf6.html")
)

<h2> Figure 3: ahrf6 fixed effects </h2>

In [ ]:
ahrf6_fe_plot <- create_forest_plot(
  data = ahrf6_fe,
  main_breaks = c(0, 1, 2, 3, 4, 5),
  main_limits = c(0, 5),
  year_breaks = c(0, 1, 2, 3, 4, 5, 6),
  year_limits = c(0, 6),
  filename = file.path(figures_dir, "figure3_ahrf6_fixed_effects.tiff")
)

<h2> Get random effects <h2>

In [ ]:
ahrf6_re <- lapply(ahrf6_model, extract_random_effects)

<h2> Figure 2. Plot proportion set tidal volume and mean provider AOR </h2>

In [ ]:
primary_a = ltvv6_proportion_plot(ahrf6_data[[1]])
primary_b = random_effect_aor(
    data = ahrf6_re[[1]],
    re_group_name = 'prov_npi_shifted',
    aor_col_name = 'AOR',
    aor_ci_low_name = 'CI_lower',
    aor_ci_up_name = 'CI_upper',
    plot_title = 'Variation in Provider LTVV Adherence (AHRF-6)',
    x_axis_name = 'Provider',
    figure_name = file.path(figures_dir, "ahrf6_aor.tiff")
)
primary_combined = (primary_a + primary_b) + plot_annotation(tag_levels = 'A')
ggsave(file.path(figures_dir, "figure2_ahrf6_two_panel.tiff"), plot = primary_combined, dpi = 600, width = 12, height = 6)

In [ ]:
# Task 17: Figure 2 exact statistics — providers never adherent and >50% adherent
prov_stats_fig2 <- ahrf6_data[[1]] %>%
  dplyr::filter(!is.na(ltvv_6)) %>%
  dplyr::group_by(prov_npi_shifted) %>%
  dplyr::summarise(
    n_days       = dplyr::n(),
    n_adherent   = sum(ltvv_6 == "1"),
    pct_adherent = n_adherent / n_days * 100,
    .groups = "drop"
  )

n_total    <- nrow(prov_stats_fig2)
n_never    <- sum(prov_stats_fig2$pct_adherent == 0)
n_majority <- sum(prov_stats_fig2$pct_adherent > 50)

cat(sprintf(
  "Task 17 - Figure 2 exact statistics (persistent AHRF cohort, LTVV <=6 cc/kg):\n  Never adherent (0%% patient-days): %d of %d providers [%.1f%%]\n  >50%% adherent: %d of %d providers [%.1f%%]\n",
  n_never,    n_total, n_never    / n_total * 100,
  n_majority, n_total, n_majority / n_total * 100
))

In [ ]:
# Task 17 (Day-1 variant): provider adherence on initial vent day only — AHRF-6 cohort
# Complements the overall Figure 2 stats; intended for manuscript intro
prov_stats_day1 <- initial_ahrf_data[[1]] %>%
  dplyr::filter(!is.na(ltvv_6)) %>%
  dplyr::group_by(prov_npi_shifted) %>%
  dplyr::summarise(
    n_days       = dplyr::n(),
    n_adherent   = sum(ltvv_6 == "1"),
    pct_adherent = n_adherent / n_days * 100,
    .groups = "drop"
  )

n_total_d1    <- nrow(prov_stats_day1)
n_never_d1    <- sum(prov_stats_day1$pct_adherent == 0)
n_majority_d1 <- sum(prov_stats_day1$pct_adherent > 50)

cat(sprintf(
  paste0(
    "Task 17 (Day-1 only) - AHRF-6 cohort:\n",
    "  Never adherent (0%% of day-1 encounters): %d of %d providers [%.1f%%]\n",
    "  >50%% adherent on day 1:                 %d of %d providers [%.1f%%]\n"
  ),
  n_never_d1,    n_total_d1, n_never_d1    / n_total_d1 * 100,
  n_majority_d1, n_total_d1, n_majority_d1 / n_total_d1 * 100
))


In [ ]:
summarize_model(ahrf6_model, ahrf6_data[[1]], 'ltvv_6', 'ahrf-6 Model')

<h2> ICU Type Adjustment: Before/After MOR and ICC </h2>

In [ ]:
# Before/after ICU-type adjustment: MOR and ICC comparison (Task 2 output)
ahrf6_no_icu_vars <- setdiff(ahrf6_vars, "icu_type_5cat")
ahrf6_no_icu_model <- fit_glmer_model(ahrf_data, ahrf6_no_icu_vars, "ltvv_6")

summary_no_icu   <- summarize_model(ahrf6_no_icu_model, ahrf_data[[1]], "ltvv_6", "ahrf-6 Without ICU Type")
summary_with_icu <- summarize_model(ahrf6_model,         ahrf_data[[1]], "ltvv_6", "ahrf-6 With ICU Type")
create_model_summary_html(list(summary_no_icu, summary_with_icu),
                          output_file = file.path(figures_dir, "ahrf6_icu_type_comparison.html"))


<h1> e-Supplement: Task 3 — ICU Type 5-Category Distribution </h1>
Reports frequency distribution of icu_type_5cat in the AHRF cohort.
The raw ADT composite (icu_composite) is superseded by the agreed 5-category mapping (meeting 2026-06-12).

In [ ]:
# Task 3: ICU type 5-category frequency distribution in AHRF cohort
# icu_composite secondary model removed — icu_type_5cat is the agreed variable (meeting 2026-06-12)
cat("--- ICU Type (5-category) Frequency Distribution ---\n")
icu_freq <- sort(table(ahrf_data[[1]]$icu_type_5cat), decreasing = TRUE)
icu_pct  <- round(prop.table(icu_freq) * 100, 1)
icu_dist <- data.frame(
  icu_type = names(icu_freq),
  n        = as.integer(icu_freq),
  pct      = as.numeric(icu_pct)
)
print(icu_dist)

# Provider count per ICU type (from first imputed dataset)
cat("\n--- Providers per ICU type (n unique providers) ---\n")
prov_by_icu <- ahrf_data[[1]] |>
  dplyr::group_by(icu_type_5cat) |>
  dplyr::summarise(n_providers = dplyr::n_distinct(prov_npi_shifted),
                   n_days      = dplyr::n(),
                   .groups = "drop")
print(prov_by_icu)

<h1> e-Supplement: Task 9 — Complete-Case Sensitivity Analysis </h1>
Run only if any model variable exceeds 20% missingness (see etable_task9_missingness.xlsx).
Compares main MICE-imputed ahrf6 model against complete-case refit.

In [ ]:
# Complete-case sensitivity analysis (Task 9)
# Identifies rows complete in the original pre-imputation AHRF data; filters imputed datasets
# by hospitalization ID so positional row-index drift cannot cause silent misalignment.
# era is derived from recorded_year and is never missing;
# substitute recorded_year so the pre-imputation dataframe (which predates era creation)
# can be used for complete-case row identification
model_vars_cc <- c(setdiff(ahrf6_vars, "era"), "recorded_year", "ltvv_6", "prov_npi_shifted")

complete_rows <- complete.cases(ahrf_data_preimpute[, model_vars_cc])
complete_ids  <- ahrf_data_preimpute$hospitalizations_joined_id[complete_rows]

cc_data_list <- lapply(ahrf_data, function(df) {
  df[df$hospitalizations_joined_id %in% complete_ids, ]
})
cc_data <- as.mitml.list(cc_data_list)

cat(sprintf("Complete-case n: %d of %d rows (%.1f%% retained)\n",
    length(complete_ids), nrow(ahrf_data_preimpute),
    100 * length(complete_ids) / nrow(ahrf_data_preimpute)))

message("Fitting complete-case model...")
ahrf6_cc_model <- fit_glmer_model(cc_data, ahrf6_vars, "ltvv_6")

summary_main <- summarize_model(ahrf6_model,    ahrf_data[[1]], "ltvv_6", "ahrf-6 Main (MICE Imputed)")
summary_cc   <- summarize_model(ahrf6_cc_model, cc_data[[1]],   "ltvv_6", "ahrf-6 Complete Case")
create_model_summary_html(list(summary_main, summary_cc),
                          output_file = file.path(figures_dir, "ahrf6_task9_complete_case_comparison.html"))


# e-Supplement: Task 4 — TeleICU-Excluded Sensitivity Analysis

Sensitivity analysis excluding TeleICU hospitals (6, 7, 8). Re-runs all three primary AHRF-6 models
(overall, day-1, subsequent) on the restricted cohort and compares MOR/ICC to primary results.

Source: R3 Major #1 — TeleICU coverage complicates attending attribution.

In [ ]:
# Task 4: TeleICU-excluded sensitivity — hospitals 6, 7, 8 removed
tele_hospitals <- c("Hospital 6", "Hospital 7", "Hospital 8")

ahrf_notele_data       <- as.mitml.list(lapply(ahrf_data,            function(df) droplevels(df[!df$hospital_id %in% tele_hospitals, ])))
initial_notele_data    <- as.mitml.list(lapply(initial_ahrf_data,    function(df) droplevels(df[!df$hospital_id %in% tele_hospitals, ])))
subsequent_notele_data <- as.mitml.list(lapply(subsequent_ahrf_data, function(df) droplevels(df[!df$hospital_id %in% tele_hospitals, ])))

cat(sprintf("No-TeleICU overall rows (imp 1): %d\n", nrow(ahrf_notele_data[[1]])))
cat(sprintf("No-TeleICU day-1 rows (imp 1):   %d\n", nrow(initial_notele_data[[1]])))
cat(sprintf("No-TeleICU subseq rows (imp 1):  %d\n", nrow(subsequent_notele_data[[1]])))

message("Fitting Task 4 overall model...")
ahrf6_notele_model <- fit_glmer_model(ahrf_notele_data, ahrf6_vars, "ltvv_6")

message("Fitting Task 4 day-1 model...")
ahrf6_notele_initial_model <- fit_glmer_model(initial_notele_data, ahrf6_vars, "ltvv_6")

message("Fitting Task 4 subsequent model...")
ahrf6_notele_subsequent_model <- fit_glmer_model(subsequent_notele_data, ahrf6_vars, "ltvv_6")

# Comparison summary is in the cell immediately after ahrf6_initial_model and ahrf6_subsequent_model are defined.


<h1> e-Figure 1. ahrf6: Secondary analysis (Initial day of MV and subsequent days of MV)
 </h1>


<h2> Fit ahrf6 initial model </h2>


In [ ]:
message("Fitting ahrf6 day-1 model...")
ahrf6_initial_model <- fit_glmer_model(initial_ahrf_data, ahrf6_vars, "ltvv_6")
message("ahrf6 day-1 model fit!")


<h2> Fit ahrf6 subsequent model </h2>


In [ ]:
message("Fitting ahrf6 subsequent model...")
ahrf6_subsequent_model <- fit_glmer_model(subsequent_ahrf_data, ahrf6_vars, "ltvv_6")
message("ahrf6 subsequent model fit!")


<h2> Get fixed effects </h2>


In [ ]:
# Initial ahrf6
ahrf6_initial_fe <- pool_fixed_effects(ahrf6_initial_model)
ahrf6_initial_table <- create_fe_table(
  fe_df = ahrf6_initial_fe,
  title = "**AHRF-6 Initial Model - Pooled Fixed Effects**",
  output_file = file.path(figures_dir, "ahrf6_initial_model.html")
)

# Subsequent ahrf6
ahrf6_subsequent_fe <- pool_fixed_effects(ahrf6_subsequent_model)
ahrf6_subsequent_table <- create_fe_table(
  fe_df = ahrf6_subsequent_fe,
  title = "**AHRF-6 Subsequent Model - Pooled Fixed Effects**",
  output_file = file.path(figures_dir, "ahrf6_subsequent_model.html")
)

<h2> Plot fixed effects </h2>


In [ ]:
# Initial ahrf6
ahrf6_initial_fe_plot <- create_forest_plot(
  data = ahrf6_initial_fe,
  main_breaks = c(0, 5, 10, 15),
  main_limits = c(0, 15),
  year_breaks = c(0, 5, 10, 15),
  year_limits = c(0, 15),
  filename = file.path(figures_dir, "ahrf6_initial_fixed_effects.tiff")
)

# Subsequent ahrf6
ahrf6_subsequent_fe_plot <- create_forest_plot(
  data = ahrf6_subsequent_fe,
  main_breaks = c(0, 1, 2, 3, 4, 5),
  main_limits = c(0, 5),
  year_breaks = c(0, 1, 2, 3, 4, 5),
  year_limits = c(0, 5),
  filename = file.path(figures_dir, "ahrf6_subsequent_fixed_effects.tiff")
)

<h2> Get random effects </h2>

In [ ]:
ahrf6_initial_re <- lapply(ahrf6_initial_model, extract_random_effects)
ahrf6_subsequent_re <- lapply(ahrf6_subsequent_model, extract_random_effects)

In [ ]:
# Task 4: Comparison summary — primary vs. TeleICU-excluded
# Placed here so ahrf6_initial_model (cell 44) and ahrf6_subsequent_model (cell 46) are already defined.
summary_notele_overall    <- summarize_model(ahrf6_notele_model,            ahrf_notele_data[[1]],       "ltvv_6", "No-TeleICU Overall")
summary_notele_initial    <- summarize_model(ahrf6_notele_initial_model,    initial_notele_data[[1]],    "ltvv_6", "No-TeleICU Day 1")
summary_notele_subsequent <- summarize_model(ahrf6_notele_subsequent_model, subsequent_notele_data[[1]], "ltvv_6", "No-TeleICU Subsequent Days")

summary_primary_overall    <- summarize_model(ahrf6_model,            ahrf_data[[1]],            "ltvv_6", "Primary Overall")
summary_primary_initial    <- summarize_model(ahrf6_initial_model,    initial_ahrf_data[[1]],    "ltvv_6", "Primary Day 1")
summary_primary_subsequent <- summarize_model(ahrf6_subsequent_model, subsequent_ahrf_data[[1]], "ltvv_6", "Primary Subsequent Days")

create_model_summary_html(
  list(summary_primary_overall,    summary_notele_overall,
       summary_primary_initial,    summary_notele_initial,
       summary_primary_subsequent, summary_notele_subsequent),
  output_file = file.path(figures_dir, "ahrf6_notele_sensitivity.html")
)


<h2> e-Figure 1. Initial and subsequent ahrf6 </h2>

In [ ]:
# Initial
initial_a = ltvv6_proportion_plot(
  initial_ahrf_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Initial, AHRF-6)'
)
initial_b = random_effect_aor(
    data = ahrf6_initial_re[[1]],
    re_group_name = 'prov_npi_shifted',
    aor_col_name = 'AOR',
    aor_ci_low_name = 'CI_lower',
    aor_ci_up_name = 'CI_upper',
    plot_title = 'Variation in Provider LTVV Adherence\n(Initial, AHRF-6)',
    x_axis_name = 'Provider',
    y_min = 0,
    y_max = 6,
    figure_name = file.path(figures_dir, "ahrf6_initial_aor.tiff")
)

# Subsequent
subsequent_a = ltvv6_proportion_plot(
  subsequent_ahrf_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Subsequent, AHRF-6)'
)
subsequent_b = random_effect_aor(
    data = ahrf6_subsequent_re[[1]],
    re_group_name = 'prov_npi_shifted',
    aor_col_name = 'AOR',
    aor_ci_low_name = 'CI_lower',
    aor_ci_up_name = 'CI_upper',
    plot_title = 'Variation in Provider LTVV Adherence\n(Subsequent, AHRF-6)',
    x_axis_name = 'Provider',
    y_min = 0,
    y_max = 6,
    figure_name = file.path(figures_dir, "ahrf6_subsequent_aor.tiff")
)

# 4 panel
initial_sub_combined <- (
  (initial_a + initial_b) /    # Row 1
  (subsequent_a + subsequent_b)  # Row 2
) + plot_annotation(tag_levels = 'A')  # Auto-labels A, B, C, D
# Save combined figure
ggsave(file.path(figures_dir, "e-Figure 1_ahrf6_initial_subsequent_4panel.tiff"), plot = initial_sub_combined, dpi = 600, width = 12, height = 12)


<h1> e-Figure 2: ahrf8 - Tidal Volume Set by Provider </h1>
(A) Variation in LPV Adherence by Provider  (B) Provider Random Effects


<h2> Fit ahrf8 model </h2>


In [ ]:
# ahrf8 uses the same covariate set as ahrf6, different outcome
ahrf8_vars <- ahrf6_vars

message("Fitting ahrf8 overall model...")
ahrf8_model <- fit_glmer_model(ahrf_data, ahrf8_vars, "ltvv_8")
message("ahrf8 model fit complete")


<h2> Get fixed effects </h2>


In [ ]:
ahrf8_fe <- pool_fixed_effects(ahrf8_model)
ahrf8_fe_table <- create_fe_table(
  fe_df = ahrf8_fe,
  title = '**AHRF-8 Model - Pooled Fixed Effects**',
  output_file = file.path(figures_dir, "ahrf8.html")
)


<h2> Plot fixed effects </h2>


In [ ]:
ahrf8_fe_plot <- create_forest_plot(
  data = ahrf8_fe,
  main_breaks = c(0, 2, 4, 6),
  main_limits = c(0, 7),
  year_breaks = c(0, 5, 10, 15, 20),
  year_limits = c(0, 20),
  filename = file.path(figures_dir, "ahrf8_fixed_effects.tiff")
)

<h2> Get random effects </h2>


In [ ]:
ahrf8_re <- lapply(ahrf8_model, extract_random_effects)

<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
ahrf8_panel_a <- ltvv8_proportion_plot(ahrf8_data[[1]], plot_title = 'Provider Tidal Volume Distribution')
ahrf8_panel_b <- random_effect_aor(
  data            = ahrf8_re[[1]],
  re_group_name   = 'prov_npi_shifted',
  aor_col_name    = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title      = 'Variation in Provider LTVV Adherence (AHRF-8)',
  x_axis_name     = 'Provider',
  figure_name     = file.path(figures_dir, "e-Figure 2_ahrf8_provider_random_effects.tiff")
)
ahrf8_two_panel <- (ahrf8_panel_a + ahrf8_panel_b) + plot_annotation(tag_levels = 'A')
ggsave(file.path(figures_dir, "e-Figure 2_ahrf8_two_panel.tiff"), plot = ahrf8_two_panel, dpi = 600, width = 12, height = 6)


In [ ]:
summarize_model(ahrf8_model, ahrf8_data[[1]], 'ltvv_8', 'ahrf-8 Model')

<h1> e-Figure 3: ahrf8 - Initial and Subsequent Days </h1>


<h2> Fit ahrf8 initial model </h2>


In [ ]:
message("Fitting initial ahrf8 model...")
initial_ahrf8_model <- fit_glmer_model(initial_ahrf_data, ahrf8_vars, "ltvv_8")
message("Initial ahrf8 model fit complete")


<h2> Fit ahrf8 subsequent model </h2>


In [ ]:
message("Fitting subsequent ahrf8 model...")
subsequent_ahrf8_model <- fit_glmer_model(subsequent_ahrf_data, ahrf8_vars, "ltvv_8")
message("Subsequent ahrf8 model fit complete")


<h2> Get fixed effects </h2>


In [ ]:
initial_ahrf8_fe    <- pool_fixed_effects(initial_ahrf8_model)
subsequent_ahrf8_fe <- pool_fixed_effects(subsequent_ahrf8_model)

initial_ahrf8_table <- create_fe_table(
  fe_df = initial_ahrf8_fe,
  title = '**Initial AHRF-8 Model - Pooled Fixed Effects**',
  output_file = file.path(figures_dir, "ahrf8_initial_model.html")
)

subsequent_ahrf8_table <- create_fe_table(
  fe_df = subsequent_ahrf8_fe,
  title = '**Subsequent AHRF-8 Model - Pooled Fixed Effects**',
  output_file = file.path(figures_dir, "ahrf8_subsequent_model.html")
)


<h2> Plot fixed effects </h2>


In [ ]:
initial_ahrf8_fe_plot <- create_forest_plot(
  data = initial_ahrf8_fe,
  main_breaks = c(0, 2, 4, 6, 8),
  main_limits = c(0, 8),
  year_breaks = c(0, 10, 20, 30, 40),
  year_limits = c(0, 45),
  filename = file.path(figures_dir, "ahrf8_initial_fixed_effects.tiff")
)

subsequent_ahrf8_fe_plot <- create_forest_plot(
  data = subsequent_ahrf8_fe,
  main_breaks = c(0, 2, 4, 6, 8),
  main_limits = c(0, 9),
  year_breaks = c(0, 5, 10, 15),
  year_limits = c(0, 15),
  filename = file.path(figures_dir, "ahrf8_subsequent_fixed_effects.tiff")
)


<h2> Get random effects </h2>


In [ ]:
initial_ahrf8_re    <- lapply(initial_ahrf8_model, extract_random_effects)
subsequent_ahrf8_re <- lapply(subsequent_ahrf8_model, extract_random_effects)


<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
initial_ahrf8_a <- ltvv8_proportion_plot(
  initial_ahrf_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Initial, AHRF-8)')
initial_ahrf8_b <- random_effect_aor(
  data = initial_ahrf8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Initial, AHRF-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 2,
  figure_name = file.path(figures_dir, "ahrf8_initial_aor.tiff")
)

subsequent_ahrf8_a <- ltvv8_proportion_plot(
  subsequent_ahrf_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Subsequent, AHRF-8)')
subsequent_ahrf8_b <- random_effect_aor(
  data = subsequent_ahrf8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Subsequent, AHRF-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 16,
  figure_name = file.path(figures_dir, "ahrf8_subsequent_aor.tiff")
)

ahrf8_initial_sub_combined <- ((initial_ahrf8_a + initial_ahrf8_b) / (subsequent_ahrf8_a + subsequent_ahrf8_b)) + plot_annotation(tag_levels = 'A')
ggsave(file.path(figures_dir, "e-Figure 3_ahrf8_initial_subsequent_4panel.tiff"), plot = ahrf8_initial_sub_combined, dpi = 600, width = 12, height = 12)

<h1> e-Figure 6: ahrf6 - SARS-CoV-2 Sensitivity </h1>
(A) Variation in LPV Adherence by Provider  (B) Provider Random Effects


<h2> Fit ahrf6 COVID sensitivity model </h2>


In [ ]:
covid_vars <- c(ahrf6_vars, "covid")

message("Fitting COVID sensitivity model...")
covid_model <- fit_glmer_model(ahrf6_data, covid_vars, "ltvv_6")
message("COVID sensitivity model fit complete")


<h2> Get fixed effects </h2>


In [ ]:
covid_fe <- pool_fixed_effects(covid_model)
covid_fe_table <- create_fe_table(
  fe_df = covid_fe,
  title = '**AHRF-6 COVID Sensitivity Model - Pooled Fixed Effects**',
  output_file = file.path(figures_dir, "ahrf6_covid_model.html")
)

<h2> Plot fixed effects </h2>


In [ ]:
covid_fe_plot <- create_forest_plot(
  data = covid_fe,
  main_breaks = c(0, 1, 2, 3, 4),
  main_limits = c(0, 4),
  year_breaks = c(0, 1, 2, 3, 4, 5),
  year_limits = c(0, 5),
  filename = file.path(figures_dir, "ahrf6_covid_fixed_effects.tiff")
)


<h2> Get random effects </h2>


In [ ]:
covid_re <- lapply(covid_model, extract_random_effects)

<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
covid_panel_a <- ltvv6_proportion_plot(
  ahrf6_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider'
)

covid_panel_b <- random_effect_aor(
  data            = covid_re[[1]],
  re_group_name   = 'prov_npi_shifted',
  aor_col_name    = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title      = 'Variation in Provider LTVV Adherence\n(AHRF-6, SARS-CoV-2)',
  x_axis_name     = 'Provider',
  figure_name     = file.path(figures_dir, "e-Figure 6_ahrf6_covid_provider_random_effects.tiff")
)
covid_two_panel <- (covid_panel_a + covid_panel_b) + plot_annotation(tag_levels = 'A')
ggsave(file.path(figures_dir, "e-Figure 6_ahrf6_covid_two_panel.tiff"), plot = covid_two_panel, dpi = 600, width = 12, height = 6)

<h1> e-Figure 4: MV8 - Secondary analysis </h1>
(A) Variation in LPV Adherence by Provider  (B) Provider Random Effects


<h2> Fit MV8 model </h2>


In [ ]:
# MV-8 uses the same covariate set as ahrf6, different cohort and outcome
mv8_vars <- ahrf6_vars

message("Fitting MV-8 overall model...")
mv8_model <- fit_glmer_model(mv_data, mv8_vars, "ltvv_8")
message("MV8 model fit complete")


<h2> Get fixed effects </h2>


In [ ]:
mv8_fe <- pool_fixed_effects(mv8_model)
mv8_fe_table <- create_fe_table(
  fe_df = mv8_fe,
  title = '**MV-8 Model - Pooled Fixed Effects**',
  output_file = file.path(figures_dir, "MV8.html")
)


<h2> Plot fixed effects </h2>


In [ ]:
mv8_fe_plot <- create_forest_plot(
  data = mv8_fe,
  main_breaks = c(0, 2, 4),
  main_limits = c(0, 4),
  year_breaks = c(0, 5, 10, 15, 20),
  year_limits = c(0, 20),
  filename = file.path(figures_dir, "mv8_fixed_effects.tiff")
)


<h2> Get random effects </h2>


In [ ]:
mv8_re <- lapply(mv8_model, extract_random_effects)


<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
mv8_panel_a <- ltvv8_proportion_plot(
  mv_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider')
mv8_panel_b <- random_effect_aor(
  data            = mv8_re[[1]],
  re_group_name   = 'prov_npi_shifted',
  aor_col_name    = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title      = 'Variation in Provider LTVV Adherence (MV-8)',
  x_axis_name     = 'Provider',
  figure_name     = file.path(figures_dir, "e-Figure 4_mv8_provider_random_effects.tiff")
)
mv8_two_panel <- (mv8_panel_a + mv8_panel_b) + plot_annotation(tag_levels = 'A')
ggsave(file.path(figures_dir, "e-Figure 4_mv8_two_panel.tiff"), plot = mv8_two_panel, dpi = 600, width = 12, height = 6)


<h1> e-Figure 5: MV8 - Initial and Subsequent Days </h1>


<h2> Fit MV8 initial model </h2>


In [ ]:
message("Fitting initial MV-8 model...")
initial_mv8_model <- fit_glmer_model(initial_mv_data, mv8_vars, "ltvv_8")
message("Initial MV8 model fit complete")


<h2> Fit MV8 subsequent model </h2>


In [ ]:
message("Fitting subsequent MV-8 model...")
subsequent_mv8_model <- fit_glmer_model(subsequent_mv_data, mv8_vars, "ltvv_8")
message("Subsequent MV8 model fit complete")


<h2> Get fixed effects </h2>


In [ ]:
initial_mv8_fe    <- pool_fixed_effects(initial_mv8_model)
subsequent_mv8_fe <- pool_fixed_effects(subsequent_mv8_model)

initial_mv8_table <- create_fe_table(
  fe_df = initial_mv8_fe,
  title = '**Initial MV-8 Model - Pooled Fixed Effects**',
  output_file = file.path(figures_dir, "MV8_initial_model.html")
)

subsequent_mv8_table <- create_fe_table(
  fe_df = subsequent_mv8_fe,
  title = '**Subsequent MV-8 Model - Pooled Fixed Effects**',
  output_file = file.path(figures_dir, "MV8_subsequent_model.html")
)


<h2> Plot fixed effects </h2>


In [ ]:
initial_mv8_fe_plot <- create_forest_plot(
  data = initial_mv8_fe,
  main_breaks = c(0, 2, 4, 6),
  main_limits = c(0, 6),
  year_breaks = c(0, 10, 20, 30, 40),
  year_limits = c(0, 40),
  filename = file.path(figures_dir, "mv8_initial_fixed_effects.tiff")
)

subsequent_mv8_fe_plot <- create_forest_plot(
  data = subsequent_mv8_fe,
  main_breaks = c(0, 2, 4),
  main_limits = c(0, 5),
  year_breaks = c(0, 5, 10, 15, 20),
  year_limits = c(0, 20),
  filename = file.path(figures_dir, "mv8_subsequent_fixed_effects.tiff")
)


<h2> Get random effects </h2>


In [ ]:
initial_mv8_re    <- lapply(initial_mv8_model, extract_random_effects)
subsequent_mv8_re <- lapply(subsequent_mv8_model, extract_random_effects)

<h2> Plot proportion set tidal volume and mean provider AOR </h2>


In [ ]:
initial_mv8_a <- ltvv8_proportion_plot(
  initial_mv_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Initial, MV-8)')
initial_mv8_b <- random_effect_aor(
  data = initial_mv8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Initial, MV-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 4,
  figure_name = file.path(figures_dir, "mv8_initial_aor.tiff")
)

subsequent_mv8_a <- ltvv8_proportion_plot(
  subsequent_mv_data[[1]],
  plot_title = 'Proportion of Set Tidal Volume by Provider\n(Subsequent, MV-8)')
subsequent_mv8_b <- random_effect_aor(
  data = subsequent_mv8_re[[1]],
  re_group_name = 'prov_npi_shifted',
  aor_col_name = 'AOR',
  aor_ci_low_name = 'CI_lower',
  aor_ci_up_name  = 'CI_upper',
  plot_title = 'Variation in Provider LTVV Adherence\n(Subsequent, MV-8)',
  x_axis_name = 'Provider',
  y_min = 0,
  y_max = 4,
  figure_name = file.path(figures_dir, "mv8_subsequent_aor.tiff")
)

mv8_initial_sub_combined <- ((initial_mv8_a + initial_mv8_b) / (subsequent_mv8_a + subsequent_mv8_b)) + plot_annotation(tag_levels = 'A')
ggsave(file.path(figures_dir, "e-Figure 5_mv8_initial_subsequent_4panel.tiff"), plot = mv8_initial_sub_combined, dpi = 600, width = 12, height = 12)


In [ ]:
# Summarize all models and write combined HTML report
ahrf6_summary <- summarize_model(ahrf6_model, ahrf6_data[[1]], 'ltvv_6', 'Overall')
ahrf6_initial_summary <- summarize_model(ahrf6_initial_model, initial_ahrf_data[[1]], 'ltvv_6', 'Day 1')
ahrf6_subsequent_summary <- summarize_model(ahrf6_subsequent_model, subsequent_ahrf_data[[1]], 'ltvv_6', 'Subsequent Days')
covid_summary <- summarize_model(covid_model, ahrf6_data[[1]], 'ltvv_6', 'ahrf-6 + COVID')

ahrf8_summary <- summarize_model(ahrf8_model, ahrf8_data[[1]], 'ltvv_8', 'ahrf-8 Overall')
initial_ahrf8_summary <- summarize_model(initial_ahrf8_model, initial_ahrf_data[[1]], 'ltvv_8', 'ahrf-8 Initial')
subsequent_ahrf8_summary <- summarize_model(subsequent_ahrf8_model, subsequent_ahrf_data[[1]], 'ltvv_8', 'ahrf-8 Subsequent')

mv8_summary <- summarize_model(mv8_model, mv_data[[1]], 'ltvv_8', 'MV-8 Overall')
initial_mv8_summary <- summarize_model(initial_mv8_model, initial_mv_data[[1]], 'ltvv_8', 'MV-8 Initial')
subsequent_mv8_summary <- summarize_model(subsequent_mv8_model, subsequent_mv_data[[1]], 'ltvv_8', 'MV-8 Subsequent')

model_summaries <- list(
  ahrf6_summary,
  ahrf6_initial_summary,
  ahrf6_subsequent_summary,
  covid_summary,
  ahrf8_summary,
  initial_ahrf8_summary,
  subsequent_ahrf8_summary,
  mv8_summary,
  initial_mv8_summary,
  subsequent_mv8_summary
)

create_model_summary_html(model_summaries, output_file = file.path(figures_dir, "table2_model_summary.html"))
